# Tutorial: Scenario Comparison, KPIs, and Analysis

Detailed step-by-step workflow notebook for this SD-dMFA repository.


## Audience, Prerequisites, Outcomes

**Audience**
- Analysts producing scenario insights and trade-off summaries.

**Prerequisites**
- Python 3.11+ environment for this repo.
- `pip install -e ".[dev]"` completed.
- Notebook executed from repository root or a subfolder.

**Outcomes**
- Generate comparison tables from latest runs.
- Analyze delta vs baseline and KPI ranking.
- Build quick custom comparative plots and filters.


## Outline

1. Build comparison package
2. Load and inspect comparison CSVs
3. Rank scenarios by KPI
4. Slice deltas by material-region
5. Plot and export analytical views


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from textwrap import dedent

import pandas as pd
import matplotlib.pyplot as plt

try:
    from crm_model.common.io import load_run_config
except Exception:
    load_run_config = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


In [ ]:
DRY_RUN = False
RUN_HEAVY = False
RUN_PLOTS = False
RUN_CALIBRATION = False
RUN_AUDIT = False

CONFIG = "configs/runs/mvp.yml"
EXAMPLE_VARIANT = "baseline"


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "configs").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError("Could not locate repo root from current working directory.")


def sh(cmd: str, *, cwd: Path, check: bool = True) -> subprocess.CompletedProcess | None:
    print(f"$ {cmd}")
    if DRY_RUN:
        return None
    cp = subprocess.run(cmd, cwd=str(cwd), shell=True, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout)
    if cp.stderr.strip():
        print(cp.stderr)
    if check and cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {cmd}")
    return cp


def latest_dir(base: Path) -> Path | None:
    if not base.exists():
        return None
    cands = [p for p in base.iterdir() if p.is_dir() and p.name != "_archive"]
    return sorted(cands)[-1] if cands else None


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


REPO = find_repo_root()
CONFIG_PATH = (REPO / CONFIG).resolve()
CONFIG_STEM = CONFIG_PATH.stem
print("Repo:", REPO)
print("Config:", CONFIG_PATH)


## Step 1: Build comparison package from latest runs


In [ ]:
_ = sh(f"python scripts/analysis/compare_scenarios.py --config {CONFIG}", cwd=REPO)
cmp_dir = REPO / "outputs" / "analysis" / "scenario_comparison" / CONFIG_STEM / "latest"
print("comparison dir:", cmp_dir)


## Step 2: Load comparison tables


In [ ]:
summary_cmp = load_csv(cmp_dir / "summary_comparison.csv")
delta_cmp = load_csv(cmp_dir / "delta_vs_baseline.csv")
kpi_cmp = load_csv(cmp_dir / "scenario_kpis.csv")

print("rows:", len(summary_cmp), len(delta_cmp), len(kpi_cmp))


## Step 3: KPI ranking


In [ ]:
if not kpi_cmp.empty:
    cols = [c for c in ["variant", "avg_final_stress_multiplier", "max_final_stress_multiplier", "avg_service_stress", "converged_all"] if c in kpi_cmp.columns]
    display(kpi_cmp[cols].sort_values("avg_final_stress_multiplier").reset_index(drop=True))


## Step 4: Delta cuts by material-region


In [ ]:
if not delta_cmp.empty:
    useful = [c for c in ["variant", "material", "region", "delta_service_stress", "delta_circular_supply_stress", "delta_stress_multiplier"] if c in delta_cmp.columns]
    cut = delta_cmp[useful].copy()
    display(cut.sort_values("delta_stress_multiplier").head(20))


## Step 5: Stress delta heatmap-style pivot


In [ ]:
if not delta_cmp.empty and "delta_stress_multiplier" in delta_cmp.columns:
    pvt = delta_cmp.pivot_table(
        index="variant",
        columns=["material", "region"],
        values="delta_stress_multiplier",
        aggfunc="mean",
    )
    display(pvt)


## Step 6: Quick chart - scenario average stress


In [ ]:
if not kpi_cmp.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    k = kpi_cmp.sort_values("avg_final_stress_multiplier")
    ax.bar(k["variant"], k["avg_final_stress_multiplier"])
    ax.set_title("Scenario ranking by average final stress multiplier")
    ax.set_ylabel("avg_final_stress_multiplier")
    ax.tick_params(axis="x", rotation=70)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()


## Pitfalls

- Comparing scenarios from mixed run timestamps can mislead; always rebuild compare package first.
- Verify `converged_all` before interpreting KPI differences.


## Exercises

1. Repeat this workflow with `CONFIG=configs/runs/r-strategies.yml`.
2. Record one thing that changed and why.
3. Add one guardrail/check specific to your team workflow.


In [ ]:
# Exercise answer scaffold
pass
